This demo showcases how to build and run a **Flotorch CrewAI Agent** with two custom tools for web research and text analysis.


In [7]:
# Install required dependencies
%pip install requests beautifulsoup4 crewai crewai_tools crewai_tools[mcp] -q


180.31s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import requests
from bs4 import BeautifulSoup
from flotorch.crewai.agent import FlotorchCrewAIAgent
from flotorch.crewai.sessions import FlotorchCrewAISession
from crewai.memory.short_term.short_term_memory import ShortTermMemory
from crewai import Crew
from crewai.tools import tool
from crewai.tools.base_tool import BaseTool

In [9]:
# Configuration
FLOTORCH_GATEWAY_BASE_URL = "https://dev-gateway.flotorch.cloud"
FLOTORCH_API_KEY = "sk_XKtu651TxvB8C9/asbeXoi9DbuMKWlnSooh1Yt5sjr0=_YTBhZDdkNzYtNGZiZi00MzM4LThiZmQtZDFhNzE5NDZjNDNk_ZTkwN2RkYjEtNWYxYy00Y2ZiLTg3ZjktMWRlMTYyNGYyMmIw"

## Tool Implementations


In [10]:
@tool
def web_scraper(url: str) -> str:
    """Scrape content from a given URL and return cleaned text."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()
        
        # Get text content
        text = soup.get_text()
        
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = '\n'.join(chunk for chunk in chunks if chunk)
        
        # Limit text length to avoid overwhelming the agent
        return text[:3000] + "..." if len(text) > 3000 else text
        
    except Exception as e:
        return f"Error scraping URL: {str(e)}"


In [11]:
@tool
def text_analyzer(text: str, include_keywords: bool = True) -> str:
    """Analyze text and provide key statistics and insights."""
    try:
        # Basic statistics
        words = text.split()
        word_count = len(words)
        char_count = len(text)
        sentence_count = len([s for s in text.split('.') if s.strip()])
        avg_word_length = sum(len(word) for word in words) / len(words) if words else 0
        
        # Build analysis report
        analysis = f"""Text Analysis Summary:
- Word Count: {word_count:,}
- Character Count: {char_count:,} 
- Sentence Count: {sentence_count:,}
- Average Word Length: {avg_word_length:.1f} characters"""
        
        # Add keywords if requested
        if include_keywords and words:
            # Find top 5 most common meaningful words
            clean_words = []
            for word in words:
                clean_word = word.lower().strip('.,!?;:"()[]{}').replace("'", "")
                if len(clean_word) > 3 and clean_word.isalpha():
                    clean_words.append(clean_word)
            
            if clean_words:
                word_freq = {}
                for word in clean_words:
                    word_freq[word] = word_freq.get(word, 0) + 1
                
                top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:5]
                keywords = ", ".join([word for word, count in top_words])
                analysis += f"\n- Top Keywords: {keywords}"
        
        return analysis
        
    except Exception as e:
        return f"Error analyzing text: {str(e)}"


## Web Research Analyst Agent


In [12]:

os.environ["FLOTORCH_ENABLE_SDK_TRACING"] = str(True).lower()

agent_name = "web-research-analyst"
agent_client = FlotorchCrewAIAgent(
    agent_name =agent_name,
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_GATEWAY_BASE_URL,
    custom_tools= [web_scraper, text_analyzer]
    )

agent = agent_client.get_agent()
task = agent_client.get_task()
short_term_memory = FlotorchCrewAISession(
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_GATEWAY_BASE_URL
    )
session_service = ShortTermMemory(storage = short_term_memory)
crew = Crew(
    agents = [agent],
    tasks = [task],
    short_term_memory = session_service,
    verbose = False
)

2025-09-26 14:27:12 - opentelemetry.trace - WARNING - Overriding of current TracerProvider is not allowed


2025-09-26 14:27:13 - flotorch.sdk.llm - INFO - FlotorchLLM initialized (model_id=flotorch/haiku-long:latest, base_url=https://dev-gateway.flotorch.cloud)


In [13]:
query = """Research and analyze content from the provided website:
1. Use the Web Scraper tool to extract content from the website
2. Use the Text Analyzer tool to analyze the scraped content  
3. Provide a comprehensive summary that includes:
    - Key information found on the website
    - Text analysis statistics
    - Main insights and takeaways"""

response = crew.kickoff(inputs = {"query":query, "url": "https://www.flotorch.ai/blogs/crewai-financial-agent-with-flotorch"})
print(f"Results: {response}")


===================START TRACING=====================
🔧 Crew: crew [TRACE_ID: de65b996f6f3c4c4e744714273f40bde]
===================START TRACING=====================
🔧 Crew: crew [TRACE_ID: 7a43bc21e05feb76e5ea77684ea4169f]
🔧 Task:  [SPAN_ID: a2f393c3421a8cd1]
🔧 Task:  [SPAN_ID: 96d31b75027bdb60]
2025-09-26 14:27:17 - opentelemetry.attributes - WARNING - Invalid type UUID for attribute 'memory.retrieval.task_id' value. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or a sequence of those types
🔧 Memory Retrieval: 06fc441f-07c5-4ac4-8071-b296c76e2ab4 [SPAN_ID: bd65ac0f5d8f9bd8]
2025-09-26 14:27:17 - opentelemetry.attributes - WARNING - Invalid type UUID for attribute 'memory.retrieval.task_id' value. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or a sequence of those types
🔧 Memory Retrieval: 06fc441f-07c5-4ac4-8071-b296c76e2ab4 [SPAN_ID: 674af83144d59573]
🔧 Memory Query: Your goal is to:
1. Use the We... [SPAN_ID: 96c0f9d0b70a9dfd]
🔧 Memory Query: Your goal is to:

In [8]:
from flotorch_eval.agent_eval.core.client import FlotorchEvalClient
client = FlotorchEvalClient(
    api_key=FLOTORCH_API_KEY,
    base_url=FLOTORCH_GATEWAY_BASE_URL,
    default_evaluator="flotorch/haiku-long") # Setting a default evaluator for all metrics that require an LLM.

trace_id = "c5ee62b092a101aeed02abdc10b46729"
traces = client.fetch_traces(trace_id)
print(f"Traces: {traces}")

Traces: {'resourceSpans': [{'resource': {'attributes': [{'key': 'service.version', 'value': {'stringValue': '1.0.0'}}, {'key': 'telemetry.sdk.language', 'value': {'stringValue': 'python'}}, {'key': 'telemetry.sdk.name', 'value': {'stringValue': 'opentelemetry'}}, {'key': 'telemetry.sdk.version', 'value': {'stringValue': '1.36.0'}}, {'key': 'service.name', 'value': {'stringValue': 'flotorch-gateway'}}]}, 'scopeSpans': [{'scope': {'name': 'flotorch-gateway'}, 'spans': [{'traceId': 'xe5isJKhAa7tAqvcELRnKQ==', 'spanId': 'nORhVBFfh1o=', 'name': 'Crew: crew', 'kind': 'SPAN_KIND_INTERNAL', 'startTimeUnixNano': '1758537435532699432', 'endTimeUnixNano': '1758537452254701274', 'attributes': [{'key': 'gen_ai.operation.name', 'value': {'stringValue': 'chat'}}, {'key': 'gen_ai.system', 'value': {'stringValue': 'flotorch'}}, {'key': 'agent_name', 'value': {'stringValue': 'crew'}}, {'key': 'output_type', 'value': {'stringValue': 'text'}}, {'key': 'crewai.crew.name', 'value': {'stringValue': 'crew'}},